In [8]:
from app import models, db_utils, crud, rating
from app.codeforces_api import cf_api
from faker import Faker
import random
import numpy as np

In [9]:
db = db_utils.SessionLocal()

In [3]:
candidate_contests = [
    2111,
    2071,
    2066,
    2048
]

In [15]:
users = db.query(models.User).all()
admin_users = [
    'negative-xp',
    'roomTemperatureIQ'
]
memberships = []
for usr in users:
    memberships.append(
        models.GroupMembership(
            user_id = usr.user_id,
            group_id = "some_group",
            role = "admin" if usr.user_id in admin_users else "user",
            cf_handle = usr.user_id
        )
    )

In [15]:
db.query(models.User).filter(models.User.user_id == "negative-xp").first().__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x113cb2f30>,
 'user_id': 'negative-xp',
 'codechef_handle': None,
 'email_id': 'negative-xp@example.com',
 'hashed_password': '8/qCEzUWTVn40dymTLfsgZDSJD0kySJgHmQpfxiWBnTzsyJkfPT9FyJgLY0wyZfV',
 'cf_handle': 'negative-xp',
 'role': <Role.admin: 'admin'>,
 'atcoder_handle': None,
 'twitter_handle': None,
 'is_registered': True,
 'timestamp': datetime.datetime(2025, 6, 6, 5, 51, 48, 572818)}

In [9]:
crud.fetch_and_add_contest_to_db_from_cf(db, "2111")

In [20]:
memberships = db.query(models.GroupMembership).all()
len(memberships)

75621

In [10]:
# add a bunch of contest participations

participations = []

for m in memberships:
    for c in candidate_contests:
        participations.append(
            models.ContestParticipation(
                user_id = m.user_id,
                group_id = m.group_id,
                contest_id = f"cf_{c}",
                cf_handle = m.cf_handle,
                rating_before = m.user_group_rating
            )
        )

db.add_all(participations)
db.commit()

NameError: name 'memberships' is not defined

In [53]:
standings = {}
for c in candidate_contests:
    standings[c] = cf_api.contest_standings(c)

In [3]:
cf_api.fetch_upcoming_contests()

[{'id': 2119,
  'name': 'Codeforces Round (Div. 2)',
  'type': 'CF',
  'phase': 'BEFORE',
  'frozen': False,
  'durationSeconds': 7200,
  'startTimeSeconds': 1751726100,
  'relativeTimeSeconds': -2437812},
 {'id': 2112,
  'name': 'Educational Codeforces Round 180 (Rated for Div. 2)',
  'type': 'ICPC',
  'phase': 'BEFORE',
  'frozen': False,
  'durationSeconds': 7200,
  'startTimeSeconds': 1750689300,
  'relativeTimeSeconds': -1401012},
 {'id': 2113,
  'name': 'Codeforces Round (Div. 2)',
  'type': 'CF',
  'phase': 'BEFORE',
  'frozen': False,
  'durationSeconds': 7200,
  'startTimeSeconds': 1749978300,
  'relativeTimeSeconds': -690012},
 {'id': 2118,
  'name': 'Codeforces Round (Div. 2)',
  'type': 'CF',
  'phase': 'BEFORE',
  'frozen': False,
  'durationSeconds': 7200,
  'startTimeSeconds': 1749738900,
  'relativeTimeSeconds': -450612},
 {'id': 2117,
  'name': 'Codeforces Round 1029 (Div. 3)',
  'type': 'ICPC',
  'phase': 'BEFORE',
  'frozen': False,
  'durationSeconds': 8100,
  'star

In [3]:
crud.update_upcoming_contests(db)

In [6]:
c = db.query(models.Contest).all()
c

[<Contest(id=c2093, name=Codeforces Round 1016 (Div. 3))>,
 <Contest(id=c2094, name=Codeforces Round 1017 (Div. 4))>,
 <Contest(id=c2096, name=Neowise Labs Contest 1 (Codeforces Round 1018, Div. 1 + Div. 2))>,
 <Contest(id=c2097, name=Codeforces Round 1021 (Div. 1))>,
 <Contest(id=c2098, name=Codeforces Round 1021 (Div. 2))>,
 <Contest(id=c2101, name=Codeforces Round 1024 (Div. 1))>,
 <Contest(id=c2103, name=Codeforces Round 1019 (Div. 2))>,
 <Contest(id=c2104, name=Educational Codeforces Round 178 (Rated for Div. 2))>,
 <Contest(id=c2106, name=Codeforces Round 1020 (Div. 3))>,
 <Contest(id=c2107, name=Codeforces Round 1023 (Div. 2))>,
 <Contest(id=c2108, name=Codeforces Round 1022 (Div. 2))>,
 <Contest(id=c2109, name=Codeforces Round 1025 (Div. 2))>,
 <Contest(id=c2110, name=Codeforces Round 1026 (Div. 2))>,
 <Contest(id=c2114, name=Codeforces Round 1027 (Div. 3))>,
 <Contest(id=c2115, name=Codeforces Round 1028 (Div. 1))>,
 <Contest(id=cf_1890, name=Codeforces Round 906 (Div. 2))>]

In [18]:
r = update_contest_ratings_for_group(db, "some_group", "cf_2111")

In [26]:
valid_participations = db.query(models.ContestParticipation).filter(
    models.ContestParticipation.group_id == "some_group",
    models.ContestParticipation.contest_id == "cf_2111",
    models.ContestParticipation.rank.isnot(None)
).all()

In [27]:
valid_participations

[]

In [7]:
res = sorted(res, key = lambda i: i.rank)

In [8]:
res[1].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x130829c70>,
 'group_id': 'some_group',
 'contest_id': 'cf_2111',
 'delta': 337,
 'rating_after': 1837,
 'cf_handle': 'StarSilk',
 'user_id': 'StarSilk',
 'rank': 1,
 'rating_before': 1500,
 'rating_change': 337,
 'timestamp': datetime.datetime(2025, 6, 7, 7, 31, 0, 632186)}